# 🃏 The Mind — LLMs + Aprendizaje por Refuerzo

Simulamos el juego cooperativo **The Mind** con 4 agentes LLM que:
- Se comunican libremente (sin revelar sus cartas)
- Aprenden a coordinarse mediante RL (GRPO)
- Desarrollan lenguaje emergente propio

## Hardware recomendado
| Config | Modelo | Dispositivo |
|--------|--------|-------------|
| GPU 4GB | Qwen2.5-1.5B-Instruct + 4bit | cuda |
| CPU 12GB | Qwen2.5-1.5B-Instruct | cpu |
| GPU 4GB (ligero) | Qwen2.5-0.5B-Instruct | cuda |

In [ ]:
# ── Instalación de dependencias ──────────────────────────────────────────────
# Ejecuta esta celda solo la primera vez

!pip install torch transformers peft accelerate bitsandbytes matplotlib tqdm

# Opcional: Unsloth para entrenar más rápido (GPU con CUDA)
# !pip install "unsloth[cu121-ampere-torch240] @ git+https://github.com/unslothai/unsloth.git"

# trl para utilidades adicionales de RL (opcional)
# !pip install trl

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import os
import torch

# Añade el directorio del proyecto al path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from environment import TheMindEnv
from agents import load_base_model, create_agents
from trainer import GRPOTrainer, TrainerConfig
from utils import setup_logging, TrainingMetrics, LanguageAnalyzer

setup_logging(level='INFO')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Configuración de hardware ─────────────────────────────────────────────────

# Detecta automáticamente el mejor dispositivo disponible
if torch.cuda.is_available():
    DEVICE = 'cuda'
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_4BIT = VRAM_GB < 6  # 4-bit si menos de 6GB VRAM
    print(f'Usando GPU ({VRAM_GB:.1f} GB VRAM). 4-bit: {USE_4BIT}')
else:
    DEVICE = 'cpu'
    USE_4BIT = False
    print('Usando CPU (más lento, pero funciona)')

# ── Selección del modelo ──────────────────────────────────────────────────────
# Opciones (de más ligero a más capaz):
#   'Qwen/Qwen2.5-0.5B-Instruct'  — ~1GB RAM/VRAM (muy rápido)
#   'Qwen/Qwen2.5-1.5B-Instruct'  — ~3GB RAM/VRAM (recomendado)
#   'Qwen/Qwen2.5-3B-Instruct'    — ~6GB RAM (solo si tienes suficiente)
#   'microsoft/Phi-3-mini-4k-instruct' — ~8GB RAM (alternativa)

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

# Para 4GB VRAM usa 0.5B:
# MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

In [ ]:
# ── [OPCIONAL] Acelerar con Unsloth ──────────────────────────────────────────
# Descomenta si tienes Unsloth instalado (solo GPU)

USE_UNSLOTH = False

if USE_UNSLOTH and DEVICE == 'cuda':
    from unsloth import FastLanguageModel
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=USE_4BIT,
    )
    base_model = FastLanguageModel.get_peft_model(
        base_model,
        r=8,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        lora_alpha=32,
        lora_dropout=0.1,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    print('Modelo cargado con Unsloth ✓')
else:
    print('Cargando modelo estándar (sin Unsloth)...')

In [ ]:
# ── Carga del modelo base ─────────────────────────────────────────────────────

if not USE_UNSLOTH:
    base_model, tokenizer = load_base_model(
        model_name=MODEL_NAME,
        device='auto' if DEVICE == 'cuda' else 'cpu',
        use_4bit=USE_4BIT,
        use_flash_attention=False,  # ponlo True si tienes GPU Ampere+
    )
    print(f'Modelo {MODEL_NAME} cargado ✓')

print(f'\nParámetros totales: {base_model.num_parameters():,}')

In [ ]:
# ── Crear agentes ─────────────────────────────────────────────────────────────
# shared_lora=True:  todos aprenden juntos (un solo adaptador)
# shared_lora=False: cada agente tiene su propio adaptador LoRA
#                    (más memoria, pero permite especialización)

NUM_PLAYERS = 4
SHARED_LORA = True  # recomendado para 4GB VRAM
LORA_R = 8          # rango LoRA. Más alto = más parámetros (max 16 para hardware limitado)

agents = create_agents(
    model=base_model,
    tokenizer=tokenizer,
    num_players=NUM_PLAYERS,
    device=DEVICE,
    lora_r=LORA_R,
    shared_lora=SHARED_LORA,
)

print(f'{len(agents)} agentes creados ✓')
print(f'LoRA compartida: {SHARED_LORA}')
agents[0].model.print_trainable_parameters()

In [ ]:
# ── Test rápido: una partida sin entrenamiento ─────────────────────────────────
print('Ejecutando partida de prueba...')

env = TheMindEnv(num_players=NUM_PLAYERS)
state = env.reset(level=1)

print(f'Manos iniciales:')
for pid, hand in state.hands.items():
    print(f'  Jugador {pid}: {hand}')

# Probar un turno
obs = env.get_observation(0)
decision = agents[0].generate_action(obs)

print(f'\nDecisión del agente 0:')
print(f'  Mensaje:   {decision["message"]}')
print(f'  Acción:    {decision["action"]}')
print(f'  Razonamiento: {decision["reasoning"][:100]}...')

In [ ]:
# ── Configuración del entrenamiento ───────────────────────────────────────────

config = TrainerConfig(
    num_episodes=200,          # empieza con 200 para ver si funciona
    num_levels=2,              # niveles 1 y 2
    group_size=4,              # 4 juegos por actualización GRPO
    lr=1e-5,                   # LR conservador para LLMs pequeños
    kl_coeff=0.01,
    clip_ratio=0.2,
    entropy_bonus=0.01,
    warmup_episodes=20,        # los primeros 20 ep solo exploración
    max_turns_per_episode=100,
    messages_per_turn=True,
    device=DEVICE,
    accumulate_grad_steps=4,   # simula batch de 4 sin más memoria
    checkpoint_every=50,
)

# Para un entrenamiento más largo y serio:
# config.num_episodes = 1000
# config.num_levels = 3

print('Configuración lista ✓')
print(f'  Episodios: {config.num_episodes}')
print(f'  LR: {config.lr}')
print(f'  Group size (GRPO): {config.group_size}')

In [ ]:
# ── Inicializar entrenador ────────────────────────────────────────────────────

metrics = TrainingMetrics()
lang_analyzer = LanguageAnalyzer()

trainer = GRPOTrainer(
    agents=agents,
    env=env,
    config=config,
)

print('Entrenador GRPO listo ✓')

In [ ]:
# ── ENTRENAMIENTO ─────────────────────────────────────────────────────────────
# ¡Esta celda puede tardar bastante!
#   - CPU 12GB Qwen2.5-1.5B: ~30-60 seg/episodio
#   - GPU 4GB Qwen2.5-1.5B:  ~5-15 seg/episodio

print('Iniciando entrenamiento...')
print('(Ctrl+C para detener y ver resultados parciales)')

try:
    trainer.train(
        metrics=metrics,
        lang_analyzer=lang_analyzer,
        verbose=True,
    )
except KeyboardInterrupt:
    print('\nEntrenamiento interrumpido por el usuario.')

In [ ]:
# ── Análisis de resultados ────────────────────────────────────────────────────

metrics.print_summary()
metrics.plot('training_metrics_final.png')

lang_analyzer.print_report()
lang_analyzer.save_log('language_log.json')

In [ ]:
# ── Ver ejemplos de mensajes recientes ───────────────────────────────────────
import json

print('Últimos 20 mensajes de los agentes:\n')
for msg in lang_analyzer.message_log[-20:]:
    print(f'  Ep {msg["episode"]:3d} | J{msg["player"]}: {msg["message"]}')

In [ ]:
# ── Vocabulario emergente por fases ───────────────────────────────────────────

vocab = lang_analyzer.get_vocabulary_evolution(n_bins=3)

fase_names = ['Inicio', 'Medio', 'Final']
for i, (bin_idx, words) in enumerate(vocab.items()):
    print(f'\nFase {fase_names[min(i, 2)]}')
    print('Top 10 palabras:')
    for word, count in words[:10]:
        bar = '█' * min(count, 30)
        print(f'  {word:15s} {bar} ({count})')

In [ ]:
# ── Jugar una partida con los agentes entrenados ──────────────────────────────

print('Partida de demostración con agentes entrenados')
print('='*60)

state = env.reset(level=2)  # nivel 2: 2 cartas por jugador

print('Manos (secretas en el juego real):')
for pid, hand in state.hands.items():
    print(f'  J{pid}: {hand}')
print()

turn = 0
max_turns = 50

while not env.is_done() and turn < max_turns:
    turn += 1
    print(f'--- Turno {turn} | Mesa: {state.table_top} | Vidas: {state.lives} ---')

    for agent in agents:
        if not state.hands[agent.player_id]:
            continue

        obs = env.get_observation(agent.player_id)
        decision = agent.generate_action(obs)

        if decision['message']:
            env.send_message(agent.player_id, decision['message'])
            print(f'  J{agent.player_id} dice: "{decision["message"]}')

        if decision['action'] == 'play':
            card = agent.get_card_to_play(obs)
            if card:
                result = env.play_card(agent.player_id, card)
                icon = '✓' if result.get('correct') else '✗'
                print(f'  J{agent.player_id} juega {card} {icon}')

        if env.is_done():
            break

print()
if state.won or state.round_over:
    print('🏆 ¡Victoria! Cartas jugadas en orden:', state.played_cards)
else:
    print('💀 Derrota. Cartas jugadas:', state.played_cards)

In [ ]:
# ── Guardar modelo final ──────────────────────────────────────────────────────

from utils import save_checkpoint

save_checkpoint(
    agents=agents,
    metrics=metrics,
    episode=config.num_episodes,
    output_dir='checkpoints',
)
print('Modelo guardado ✓')